# 제출용 추론 노트북

**이 노트북을 동결(Save Version)하여 제출합니다.**

이 노트북은 **학습된 모델을 불러와 예측만** 수행합니다.

---




## 지켜야 할 규칙


**1. 날짜를 하드코딩하지 마세요.** 예측 기간은 반드시 `PRED_DATES`를 참조해야 합니다.
운영진은 이 값만 바꿔 실행합니다. 날짜가 코드에 박혀 있으면 채점이 불가능합니다.

**2. 마지막에 `pred` DataFrame을 만드세요.** 형식은 셀 2 설명을 참고하세요.



## 사전 준비

1. 학습된 모델을 파일로 저장 → **캐글 Dataset으로 업로드** (형식 자유)
   - Public으로 설정하거나 운영진 계정에 공유
2. 우측 **Add Input** 으로 연결
   - 대회 데이터셋 (`station_list.csv`)
3. **Session options → Internet: On**

> `station_list.csv` 는 **어떤 지점을 채점하는지만** 알려줍니다.



## 1. 설정 — 수정하지 마세요

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  ★ 운영진 수정 구역 — 채점 시 아래 3줄만 교체합니다 ★               ║
# ╚═══════════════════════════════════════════════════════════════════╝
API_KEY    = ""                # 기상청 API Hub 인증키
PRED_START = "20260624"        # 예측 시작일 (YYYYMMDD)
PRED_END   = "20260630"        # 예측 종료일 (YYYYMMDD)
# ═══════════════════════════════════════════════════════════════════════

import os, sys, glob
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

if not API_KEY:
    sys.exit("API_KEY를 입력하세요. (Kaggle Secrets는 사용할 수 없습니다)")

# 예측 대상 날짜 ── 자유 구현 영역에서 이 변수를 사용하세요
PRED_DATES = []
_d, _last = (datetime.strptime(s, "%Y%m%d").date() for s in (PRED_START, PRED_END))
while _d <= _last:
    PRED_DATES.append(_d)
    _d += timedelta(days=1)

# 평가 대상 지점 ── station_list.csv 에 정의된 96개
_hits = glob.glob("/kaggle/input/**/station_list.csv", recursive=True)
if not _hits:
    sys.exit("station_list.csv 를 찾을 수 없습니다. Add Input을 확인하세요.")
STATIONS = sorted(pd.read_csv(_hits[0])["STN_ID"].dropna().astype(int).unique().tolist())

print(f"예측 기간 : {PRED_START} ~ {PRED_END}  ({len(PRED_DATES)}일)")
print(f"평가 지점 : {len(STATIONS)}개")


## 2. 자유 구현 ★ 여기만 채우세요 ★

### `pred` 반환 형식

| 컬럼 | 내용 |
|---|---|
| `Date` | 정수 `YYYYMMDD` |
| `STN_ID` | 정수 지점번호 |
| `TA` | 기온 예측값 (°C) |
| `HM` | 습도 예측값 (%) |

`PRED_DATES` × `STATIONS` 의 **모든 조합**이 정확히 한 번씩 있어야 하고, 결측이 없어야 합니다.

### 유의사항

- **학습 코드를 넣지 마세요.** 추론 전용입니다. 실행 시간이 비정상적으로 길면 재학습으로 간주될 수 있습니다.
- **무작위성을 제거하세요.** 같은 입력에 항상 같은 출력이 나와야 합니다.
- **학습 때와 동일하게 피처를 만드세요.** 순서나 전처리가 달라지면 예측이 어긋납니다.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  ↓↓↓ 자유 구현 영역 ↓↓↓
#
#  
#  셀을 추가로 나누셔도 됩니다.
# ═══════════════════════════════════════════════════════════════════

# 1) 학습 산출물 로드
#    예: bundle = pickle.load(open(glob.glob("/kaggle/input/**/model.pkl",
#                                            recursive=True)[0], "rb"))

# 2) PRED_DATES 를 순회하며 위성 데이터 수집 · 피처 생성 · 예측
#    위성 API: https://apihub.kma.go.kr/api/typ05/api/GK2A/LE1B/{채널}/{영역}/data
#              ?date={YYYYMMDDHHMM}&authKey={API_KEY}

# 3) 결과를 pred 에 담습니다 ★ 이 변수명은 고정 ★
pred = pd.DataFrame({
    "Date":   np.repeat([int(d.strftime("%Y%m%d")) for d in PRED_DATES], len(STATIONS)),
    "STN_ID": np.tile(STATIONS, len(PRED_DATES)),
    "TA":     20.0,     # ← 실제 예측값으로 교체
    "HM":     70.0,     # ← 실제 예측값으로 교체
})


## 3. 제출 파일 생성 — 수정하지 마세요

In [ ]:
# ── pred 검증 ─────────────────────────────────────────────
if "pred" not in dir():
    sys.exit("자유 구현 영역에서 'pred' DataFrame을 만들어야 합니다.")
if not isinstance(pred, pd.DataFrame):
    sys.exit(f"pred 는 DataFrame 이어야 합니다 (현재: {type(pred).__name__})")

need = {"Date", "STN_ID", "TA", "HM"}
if not need <= set(pred.columns):
    sys.exit(f"pred 컬럼 부족: {sorted(need - set(pred.columns))}")

# 날짜 × 지점의 모든 조합이 정확히 한 번씩 있어야 합니다
_want = {(int(d.strftime("%Y%m%d")), s) for d in PRED_DATES for s in STATIONS}
_got  = [(int(a), int(b)) for a, b in
         zip(pred["Date"].astype(int), pred["STN_ID"].astype(int))]
if len(_got) != len(set(_got)):
    sys.exit("pred 에 중복된 (Date, STN_ID) 조합이 있습니다.")
if set(_got) != _want:
    miss, extra = _want - set(_got), set(_got) - _want
    sys.exit(f"pred 행 구성 오류 — 누락 {len(miss)}개, 불필요 {len(extra)}개\n"
             f"  누락 예시: {sorted(miss)[:3]}\n"
             f"  불필요 예시: {sorted(extra)[:3]}")

# ── 제출 파일 구성 ────────────────────────────────────────
submission = pd.DataFrame({
    "ID": pred["Date"].astype(int).astype(str) + "_"
          + pred["STN_ID"].astype(int).astype(str),
    "TA": np.clip(pd.to_numeric(pred["TA"], errors="coerce"), -50, 50).round(2),
    "HM": np.clip(pd.to_numeric(pred["HM"], errors="coerce"), 0, 100).round(2),
}).sort_values("ID").reset_index(drop=True)

if submission[["TA", "HM"]].isna().any().any():
    n = int(submission[["TA", "HM"]].isna().any(axis=1).sum())
    sys.exit(f"결측 예측값 {n}행 — 모든 행에 값이 있어야 합니다.")

out = "/kaggle/working/submission.csv"
submission.to_csv(out, index=False)

print("=" * 52)
print(f"  저장 완료: {out}")
print(f"  {len(submission)}행 = {len(STATIONS)}지점 x {len(PRED_DATES)}일")
print(f"  TA {submission.TA.min():.1f} ~ {submission.TA.max():.1f} C")
print(f"  HM {submission.HM.min():.1f} ~ {submission.HM.max():.1f} %")
print("=" * 52)
print(submission.head().to_string(index=False))


## 제출 전 점검

- [ ] 자유 구현 영역에 **날짜가 하드코딩되어 있지 않은가** (`PRED_DATES` 참조 확인)
- [ ] 자유 구현 영역에 **학습 코드가 남아 있지 않은가**
- [ ] 모델 Dataset이 Public이거나 운영진에 공유되었는가
- [ ] API 키를 코드에 직접 입력했는가 (Secrets 미사용)
- [ ] 셀 1과 셀 3을 수정하지 않았는가

### 날짜 교체 리허설

`PRED_START` / `PRED_END` 를 **다른 기간으로 바꿔** Run All 해 보세요.
운영진 채점과 똑같은 상황입니다. 여기서 에러가 나면 채점이 불가능합니다.

### 재현성 확인

Copy & Edit 로 사본을 만들어 그대로 Run All 했을 때 `submission.csv` 가
**동일한 값**으로 나오는지 확인하세요. 달라진다면 무작위성이 남아 있는 것입니다.

### 동결 및 제출

1. **Save Version → "Save & Run All (Commit)"** ("Quick Save"는 동결로 인정되지 않습니다)
2. **Versions** 탭에서 버전 번호·저장 시각 확인 (UTC 표시, KST = UTC + 9h)
3. **Share** 에서 운영진 계정을 Collaborator로 추가
